In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [3]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAI, OpenAIEmbeddings

/Users/choyoungrae/Desktop/book-study/books/RAG 마스터 랭체인으로 완성하는 LLM 서비스/rag-master-practice-260326/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
query_result = embeddings.embed_query("저는 배가 고파요")
print(query_result)

[-0.01663108356297016, -0.021850483492016792, 0.015224344097077847, -0.027161912992596626, -0.03681188449263573, 0.01172721479088068, -0.034550584852695465, -0.006731315981596708, -0.02394087240099907, -0.016815142706036568, -0.008124909363687038, 0.010826637968420982, -0.010504534468054771, -0.022626161575317383, 0.011326228268444538, -0.004890721756964922, 0.012075613252818584, -0.002912083175033331, 0.008216938935220242, -0.01613149419426918, 0.0013837324222549796, -0.014566988684237003, 0.018274471163749695, -0.012634364888072014, 0.003244047285988927, 0.006389491725713015, 0.0053311497904360294, -0.019037002697587013, -0.009235839359462261, -0.0017962227575480938, 0.03452428802847862, -0.016499612480401993, 0.00013054661394562572, 0.0031191499438136816, 0.007382097654044628, -0.0055086356587708, -0.006382917985320091, 0.00344125390984118, -0.0038849685806781054, -0.0021955659613013268, -0.02238951437175274, -0.010373063385486603, 0.01129993423819542, -0.024769140407443047, 0.01323

In [5]:
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])
df

,text
0,주식 시장이 급등했어요
1,시장 물가가 올랐어요
2,전통 시장에는 다양한 물품들을 팔아요
3,부동산 시장이 점점 더 복잡해지고 있어요
4,저는 빠른 비트를 좋아해요
5,최근 비트코인 가격이 많이 변동했어요


In [6]:
# 텍스트를 임베딩 벡터로 변환하는 함수 정의
def get_embedding(text: str):
  return embeddings.embed_query(text)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(lambda row: get_embedding(
        row.text,
    ), axis=1)

# 변환된 DataFrame 출력
df

,text,embedding
0,주식 시장이 급등했어요,"[-0.01370231993496418, -0.035045821219682693, ..."
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243..."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.017441052943468094, -0.0031565295066684484..."
4,저는 빠른 비트를 좋아해요,"[-0.03796839341521263, -0.013386794365942478, ..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.01749446988105774, -0.021561449393630028, ..."


In [7]:
def cos_sim(A: np.ndarray, B: np.ndarray):
    return dot(A, B)/(norm(A)*norm(B))

def return_answer_candidate(df: pd.DataFrame, query: str):
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = get_embedding(query)

    # DataFrame의 각 문서 임베딩과 쿼리 임베딩 간의 유사도 계산
    df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))

    # 유사도가 높은 순으로 정렬하고 상위 3개 문서 선택
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc

In [8]:
return_answer_candidate(df, "과일 값이 비싸다.")

,text,embedding,similarity
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ...",0.824548
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243...",0.814571
0,주식 시장이 급등했어요,"[-0.01370231993496418, -0.035045821219682693, ...",0.805389


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

# 데이터셋 생성
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])

# DataFrame의 각 행을 벡터로 변환
df['embedding'] = df['text'].apply(get_embedding)
df

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 54645.73it/s]


,text,embedding
0,주식 시장이 급등했어요,"[-0.028310662135481834, -0.002016831422224641,..."
1,시장 물가가 올랐어요,"[0.0039781262166798115, 0.02656499482691288, -..."
2,전통 시장에는 다양한 물품들을 팔아요,"[0.00010136908531421795, 0.0005807465640828013..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.0077268569730222225, 0.021975155919790268,..."
4,저는 빠른 비트를 좋아해요,"[-0.019412294030189514, 0.012223096564412117, ..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.026391318067908287, -0.00452756742015481, ..."


In [14]:
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.0039781262166798115, 0.02656499482691288, -...",0.642563
5,최근 비트코인 가격이 많이 변동했어요,"[-0.026391318067908287, -0.00452756742015481, ...",0.604495
0,주식 시장이 급등했어요,"[-0.028310662135481834, -0.002016831422224641,...",0.575970
